In [1]:

from pathlib import Path
import json
OUT=(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file()) / 'fusion/cnn_bilstm_attention_v1')
nb=json.loads((OUT/'01_attention_training.ipynb').read_text());exec(compile(''.join(nb['cells'][0]['source']),'definitions_only','exec'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
FUS=ROOT/'fusion/cnn_bilstm_v1'
cks={h:torch.load(OUT/'models'/f'attention_{h}s_best.pt',map_location='cpu',weights_only=False) for h in [1,2,3]}
assert all(sha(Path(p))==v for p,v in cks[1]['input_hashes'].items())
reference=json.loads((FUS/'final_fusion_report.md').read_text(encoding='utf-8').split('```json\n')[1].split('\n```')[0]);results={};plotdata={}
@torch.inference_mode()
def fusion_predict(head,z,mu,sd):
 head.eval();a=[];b=[]
 for v in ((z-mu)/sd).split(256):
  p,m=head(v.to(DEV));a.append(p.sigmoid().cpu());b.append(m.cpu())
 return torch.cat(a).numpy(),torch.cat(b).numpy()
@torch.inference_mode()
def latency(h,c,b,head,ck):
 with h5py.File(V3/'dataset'/'eew_dataset_test.h5','r') as w,h5py.File(TAB,'r') as t:
  x=torch.from_numpy(w[f'windows_{h}s_norm'][0:1][:,None,:]).to(DEV);v=torch.from_numpy(t['test'][f'{h}s/X'][MAP['test'][0]:MAP['test'][0]+1]).to(DEV)
 cm,cs,sm,ss=[ck[k].to(DEV) for k in ['cmean','cscale','smean','sscale']]
 def run():
  q=c.features(x.sign()*torch.log1p(x.abs()));ce=c.shared(torch.cat([q.mean(-1),q.amax(-1)],1));states,_=b.lstm(v);return head((ce-cm)/cs,(states-sm)/ss)
 for _ in range(10):run()
 elapsed=[]
 for _ in range(100):
  if DEV.type=='cuda':torch.cuda.synchronize()
  start=time.perf_counter();run()
  if DEV.type=='cuda':torch.cuda.synchronize()
  elapsed.append((time.perf_counter()-start)*1000)
 return {'median_ms':float(np.median(elapsed)),'p95_ms':float(np.quantile(elapsed,.95)),'repeats':100,'batch_size':1,'scope':'warm device-resident two-backbone plus attention inference; excludes I/O, tabular extraction and transfer; not streaming detection delay'}
def paired(y,target,fp,fm,bp,bm):
 rng=np.random.default_rng(2026);eq=np.flatnonzero(y==1);noise=np.flatnonzero(y==0);fdelta=[];mdelta=[]
 def f1fast(yy,p):
  yy=yy==1;q=p>=.5;tp=(yy&q).sum();return float(2*tp/(2*tp+(~yy&q).sum()+(yy&~q).sum()))
 for _ in range(500):
  a=rng.choice(eq,len(eq),replace=True);n=rng.choice(noise,len(noise),replace=True);ix=np.r_[a,n];fdelta.append(f1fast(y[ix],fp[ix])-f1fast(y[ix],bp[ix]));mdelta.append(float((np.abs(fm[a]-target[a])-np.abs(bm[a]-target[a])).mean()))
 return {'replicates':500,'seed':2026,'method':'paired stratified bootstrap; same sampled EQ/noise indices for both models','f1_difference_ci95':np.quantile(fdelta,[.025,.975]).tolist(),'mae_difference_ci95':np.quantile(mdelta,[.025,.975]).tolist()}


Exact paired inputs; frozen temporal backbone states available.


In [2]:

for h,ck in cks.items():
 c,b,cc,bc=branches(h);head=AttentionHead().to(DEV);head.load_state_dict(ck['head_state']);head.eval();va=temporal('validation',h,c,b)
 assert abs(epoch(head,loader(va,ck),ck)['total']-ck['best_validation_loss'])<1e-6
 d=temporal('test',h,c,b,True);base=encode('test',h,c,b,cc,bc,True);assert np.array_equal(d['ids'],base['ids']);y=d['y'].numpy();target=d['m'].numpy();reg=d['reg'].numpy();identity=hashlib.sha256('\n'.join(d['ids']).encode()).hexdigest()
 assert reg.sum()==1200 and (~reg).sum()==1214 and identity==reference[str(h)]['common_identity_sha256']
 p,m,w=predict_attention(head,d,ck);assert np.allclose(w.sum(1),1,atol=1e-6) and w.shape==(2414,10+10*h) and np.isfinite(w).all()
 fc=torch.load(FUS/'models'/f'fusion_{h}s_best.pt',map_location='cpu',weights_only=False);fh=FusionHead().to(DEV);fh.load_state_dict(fc['head_state']);fp,fm=fusion_predict(fh,base['z'],fc['embedding_mean'],fc['embedding_scale']);fm=fm*fc['target_scale']+fc['target_center']
 predictions={'CNN':(base['cnn_p'],base['cnn_m']),'BiLSTM':(base['bilstm_p'],base['bilstm_m']),'Fusion':(fp,fm),'Attention':(p,m)};models={}
 for name,(pp,mm) in predictions.items():
  models[name]={'fixed_05':det(y,pp),'magnitude':magnitude(target[reg],mm[reg])}
  if name!='Attention':
   assert abs(models[name]['fixed_05']['f1']-reference[str(h)]['models'][name]['fixed_05']['f1'])<1e-6
   assert abs(models[name]['magnitude']['mae']-reference[str(h)]['models'][name]['magnitude']['mae'])<1e-6
 comparisons={name:{'f1_difference':models['Attention']['fixed_05']['f1']-models[name]['fixed_05']['f1'],'mae_difference':models['Attention']['magnitude']['mae']-models[name]['magnitude']['mae'],**paired(y,target,p,m,pp,mm)} for name,(pp,mm) in predictions.items() if name!='Attention'}
 results[h]={'best_epoch':ck['best_epoch'],'common_identity_sha256':identity,'earthquakes':1200,'noise':1214,'models':models,'validation_threshold':ck['threshold'],'validation_selected_test':det(y,p,ck['threshold']),'paired_comparisons':comparisons,'latency':latency(h,c,b,head,ck),'attention_mean_entropy':float((-w*np.log(np.maximum(w,1e-12))).sum(1).mean()),'mean_attention_by_bin':w.mean(0).tolist()}
 chosen=[int(np.flatnonzero(y==0)[0]),int(np.flatnonzero(y==1)[0]),int(np.flatnonzero(target>=5)[0])]
 plotdata[h]={'actual':target[reg],'predicted':m[reg],'residual':m[reg]-target[reg],'weights':w[chosen],'ids':d['ids'][chosen].tolist(),'types':['First noise','First earthquake','First M>=5 earthquake']}
 print(f'{h}s epoch {ck["best_epoch"]}: Attention F1={models["Attention"]["fixed_05"]["f1"]:.6f}, MAE={models["Attention"]["magnitude"]["mae"]:.6f}',flush=True)


1s epoch 3: Attention F1=0.672045, MAE=0.666359
2s epoch 17: Attention F1=0.717199, MAE=0.636489
3s epoch 32: Attention F1=0.747428, MAE=0.606601


In [3]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
def save(fig,name):
 p=OUT/'plots'/name;assert not p.exists();fig.tight_layout();fig.savefig(p,dpi=300,bbox_inches='tight');plt.close(fig)
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,ck) in zip(axs,cks.items()):
 e=[q['epoch'] for q in ck['history']];ax.plot(e,[q['train']['total'] for q in ck['history']],label='Train epoch-end');ax.plot(e,[q['validation']['total'] for q in ck['history']],label='Validation');ax.axvline(ck['best_epoch'],color='black',ls=':');ax.set(title=f'{h}s',xlabel='Epoch',ylabel='BCE + masked standardized Huber');ax.legend()
save(fig,'training_validation_loss.png')
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,(h,r) in zip(axs,results.items()):
 cm=np.array(r['models']['Attention']['fixed_05']['confusion_matrix']);ax.imshow(cm,cmap='Blues');ax.set(title=f'{h}s attention; threshold .5',xticks=[0,1],yticks=[0,1],xticklabels=['Noise','EQ'],yticklabels=['Noise','EQ'],xlabel='Predicted',ylabel='Actual')
 for i in range(2):
  for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
save(fig,'confusion_matrices.png')
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,d) in zip(axs,plotdata.items()):
 ax.scatter(d['actual'],d['predicted'],s=9,alpha=.35);lo=min(d['actual'].min(),d['predicted'].min());hi=max(d['actual'].max(),d['predicted'].max());ax.plot([lo,hi],[lo,hi],'k--');ax.set(title=f'{h}s common EQ',xlabel='Actual magnitude',ylabel='Predicted magnitude')
save(fig,'actual_predicted_magnitude.png')
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,d) in zip(axs,plotdata.items()):ax.hist(d['residual'],bins=35);ax.axvline(0,color='black',ls='--');ax.set(title=f'{h}s',xlabel='Predicted minus actual',ylabel='Count')
save(fig,'magnitude_residuals.png')
for metric,name in [('f1','four_model_f1.png'),('mae','four_model_mae.png')]:
 fig,ax=plt.subplots(figsize=(8,4));x=np.arange(3)
 for offset,model in zip([-.3,-.1,.1,.3],['CNN','BiLSTM','Fusion','Attention']):ax.bar(x+offset,[results[h]['models'][model]['fixed_05' if metric=='f1' else 'magnitude'][metric] for h in [1,2,3]],width=.2,label=model)
 ax.set(xticks=x,xticklabels=['1s','2s','3s'],xlabel='Observed post-P horizon',ylabel=metric.upper(),title='Identical common test cohort'+('; threshold .5' if metric=='f1' else ''));ax.legend();save(fig,name)

fig,axs=plt.subplots(1,3,figsize=(15,4))
for ax,(h,d) in zip(axs,plotdata.items()):
 t=np.arange(10+10*h)*.1-1+.05
 for weight,label in zip(d['weights'],d['types']):ax.plot(t,weight,label=label)
 ax.axvline(0,color='black',ls=':');ax.set(title=f'{h}s observed-window attention',xlabel='Bin center relative to reference (s)',ylabel='Attention weight');ax.legend(fontsize=8)
save(fig,'temporal_attention_weights.png')
assert all(sha(Path(p))==v for p,v in cks[1]['input_hashes'].items())
lines=['# CNN + BiLSTM + temporal attention v1','', '**Scope:** synthetic oracle-P-aligned research; three models trained and evaluated.','', '## Architecture and prespecified protocol','', 'Frozen horizon-specific CNN learned 32D shared embedding plus ALL bidirectional LSTM hidden states (20/30/40 steps, 64D each). Additive attention Linear(64,32), tanh, Linear(32,1), temporal softmax, weighted sum of states; concatenate with CNN to 96D; Linear(96,32), SiLU, dropout .2 and separate binary-logit/magnitude heads. The original BiLSTM shared pooling/head is bypassed, not changed. Bidirectionality uses only the fully observed window; attention is not an online prefix-invariant representation before that cutoff.','', 'Per-channel CNN and hidden-state scaling fitted exclusively on eligible training records (hidden-state moments across records/time); fixed on validation/test. Existing seven-feature train-only normalization reused. BCE plus earthquake-only standardized Huber, AdamW lr .001/weight decay .0001, gradient clip5, seed42, maximum35 epochs, patience7, min_delta .0001. Selection minimizes full validation BCE plus earthquake mean Huber. Validation F1 threshold grid .10:.01:.90, ties closest .5 then lower. All three checkpoints saved before test inference. No test-driven tuning.','', 'Exact parent-key joins, all source/label/magnitude/mask equality and split-parent isolation checked before training. No target/arrival/ID metadata enters either network. Per-horizon pre-S masks filter train/validation; test uses shared 1,200 earthquakes and 1,214 noise. Regression includes every eligible earthquake independent of detection. All baseline fixed-.5 F1 and MAE reproduce within1e-6 on the identical ordered source-ID cohort.','', '## Selected checkpoints','', '| Horizon | Best/run epochs | Validation loss | Threshold | Parameters | Training seconds |','|---|---:|---:|---:|---:|---:|']
for h,ck in cks.items():lines.append(f"| {h}s | {ck['best_epoch']}/{len(ck['history'])} | {ck['best_validation_loss']:.8f} | {ck['threshold']:.2f} | {ck['trainable_parameters']} | {ck['training_seconds']:.2f} |")
lines+=['','Checkpoints: `models/attention_1s_best.pt`, `models/attention_2s_best.pt`, `models/attention_3s_best.pt`. These contain attention/head weights, train-fitted scalers, histories, thresholds and original dependency hashes; original backbone weights are reused without duplication. Reloaded validation losses match within1e-6.','', '## Four-model comparison (fixed 0.5 throughout)','', '| Horizon | Model | Accuracy | Precision | Recall | F1 | FPR | ROC-AUC | PR-AUC(AP) | MAE | RMSE | R2 |','|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for h,r in results.items():
 for name,v in r['models'].items():
  d=v['fixed_05'];m=v['magnitude'];lines.append(f'| {h}s | {name} | '+' | '.join(f'{d[k]:.6f}' for k in ['accuracy','precision','recall','f1','false_positive_rate','roc_auc','pr_auc_ap'])+' | '+' | '.join(f'{m[k]:.6f}' for k in ['mae','rmse','r2'])+' |')
lines+=['','## Attention operating points','','| Horizon | Threshold | Precision | Recall | F1 | FPR | Confusion matrix [noise,EQ] |','|---|---:|---:|---:|---:|---:|---|']
for h,r in results.items():
 for t,d in [(.5,r['models']['Attention']['fixed_05']),(r['validation_threshold'],r['validation_selected_test'])]:lines.append(f"| {h}s | {t:.2f} | {d['precision']:.6f} | {d['recall']:.6f} | {d['f1']:.6f} | {d['false_positive_rate']:.6f} | {d['confusion_matrix']} |")
lines+=['','Validation-selected thresholds trade precision/FPR against recall; do not compare these to fixed-.5 baseline values as equivalent policies. Complete accuracy/AUC and confusion matrices for both operating points are in the embedded results.','', '## Magnitude compression and strong-event bias','','| Horizon | Mean signed error | Pred/actual SD | Slope | M>=5 N | Underprediction % | Strong bias |','|---|---:|---:|---:|---:|---:|---:|']
for h,r in results.items():
 m=r['models']['Attention']['magnitude'];lines.append(f"| {h}s | {m['bias']:.6f} | {m['std_ratio']:.6f} | {m['slope']:.6f} | {m['strong_n']} | {m['strong_underprediction_percent']:.3f} | {m['strong_bias']:.6f} |")
lines+=['','## Paired uncertainty','','Attention-minus-baseline differences; 500 paired, class-stratified bootstrap resamples, seed2026; percentile95% intervals. Same earthquake draws for magnitude. Single-seed, exploratory, no multiplicity correction; tiny changes or intervals crossing zero do not establish improvement.','', '| Horizon | Baseline | F1 difference [95% CI] | MAE difference [95% CI] |','|---|---|---|---|']
for h,r in results.items():
 for name,d in r['paired_comparisons'].items():lines.append(f"| {h}s | {name} | {d['f1_difference']:.6f} {d['f1_difference_ci95']} | {d['mae_difference']:.6f} {d['mae_difference_ci95']} |")
robust=all(r['paired_comparisons']['Fusion']['f1_difference_ci95'][0]>0 and r['paired_comparisons']['Fusion']['mae_difference_ci95'][1]<0 for r in results.values())
lines+=['', '**Consistent improvement over non-Attention fusion:** '+('Supported on both reported metrics across horizons in these descriptive intervals only.' if robust else 'Not established across horizons and both metrics. Report mixed/worse outcomes as measured; no settings were changed after viewing test results.'),'', '## Inference timing and attention diagnostics','',f"Environment: {cks[1]['environment']}",'', '| Horizon | Warm batch-1 median ms | p95 ms |','|---|---:|---:|']
for h,r in results.items():lines.append(f"| {h}s | {r['latency']['median_ms']:.4f} | {r['latency']['p95_ms']:.4f} |")
lines+=['','Timing includes both backbones and attention/head on device-resident inputs, 10 warmups/100 repeats, synchronized GPU timing. Excludes I/O, feature extraction and transfer; not operational latency or streaming detection delay. Observation horizons are 1/2/3s post-oracle-reference with 1s preceding context.','', 'Attention figure selects the first noise, first earthquake and first M>=5 earthquake in cohort order, independently of prediction quality. Bin centers are plotted relative to the reference; weights sum to1. These distributions are diagnostics, not proof of physical feature importance. Bidirectional states contain contextual information from throughout the available window, so a high weight cannot be assigned uniquely to an isolated physical time sample.','', '## Scientific limitations','', 'Synthetic oracle-P-aligned benchmark; both branches share the same synthetic observations. Operational EEW performance, autonomous triggering and real-world generalization are unverified. High false-positive rates, magnitude compression and strong-event underprediction remain risks. Existing test cohort has been inspected in prior experiments; present comparisons are exploratory rather than fresh confirmatory evidence. Architecture/capacity differences prevent attributing all changes uniquely to attention. No further tuning, independent audit or cleanup performed.','', '## Outputs and preservation','']
for p in sorted((OUT/'plots').glob('*.png')):lines.append(f'- [plots/{p.name}](plots/{p.name}) (300 DPI)')
lines+=['',f"All {len(cks[1]['input_hashes'])} original dependencies retain exact SHA256 hashes. Both backbones remain requires_grad=False, eval mode and tensor-identical to source checkpoints after training. Writes confined to this new output directory. No datasets regenerated or copied. Hashes and environment are in each checkpoint.",'','Complete results:','','```json',json.dumps(results,indent=2),'```']
with (OUT/'final_attention_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Seven plots and report saved; originals unchanged.')


Seven plots and report saved; originals unchanged.
